# Week 4 Activity: Delay, Filters, Reverb

Complete this activity as part of your participation grade. Pending length of the lecture, you will have time in class to work. Everything you need to complete this activity can be found in this week's (or a previous week's) lecture code. For this activity, you may want to consult your Audio Tech I notes (or the Computer Music Tutorial)

In [1]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import sawtooth, square, butter, filtfilt
import matplotlib.pyplot as plt
from IPython.display import Audio, Image

## Delay
1) Create a function that will modify a passed signal by adding a delay of $m$ milliseconds to the signal.

In [ ]:
def delay_and_add(data, fs, offset_ms, dtype=None):
    
    x = np.asarray(data)
    if x.ndim != 1:
        raise ValueError("data must be a 1-D array")

    # delay in samples (rounded to nearest integer)
    D = int(round(fs * (offset_ms / 1000.0)))
    if D < 0:
        raise ValueError("offset_ms must be >= 0")

    # pick a safe dtype (avoid integer overflow when adding)
    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)  # promotes ints to float32+
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)

    pad = np.zeros(D, dtype=out_dtype)
    orig = np.concatenate([x, pad])
    delayed = np.concatenate([pad, x])

    new_s = orig + delayed
    return new_s

2) Modify the function such that you can optionally scale the amplitude of the delayed signal 

In [ ]:
def delay_and_add(data, fs, offset_ms, gain=1.0, dtype=None):
    x = np.asarray(data)
    if x.ndim != 1:
        raise ValueError("data must be a 1-D array")

    D = int(round(fs * (offset_ms / 1000.0)))
    if D < 0:
        raise ValueError("offset_ms must be >= 0")

    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)

    pad = np.zeros(D, dtype=out_dtype)
    orig = np.concatenate([x, pad])
    delayed = np.concatenate([pad, x])

    new_s = orig + gain * delayed
    return new_s

3) Modify the function so that the user can specify the number of delays to add to the original signal (and optionally, scale each subsequent delay amplitude by a factor of $1/n$

In [ ]:
def delay_and_add(data, fs, offset_ms, gain=1.0, dtype=None, num_delays=1, scale_by_1_over_n=False):
    
    x = np.asarray(data)
    if x.ndim != 1:
        raise ValueError("data must be a 1-D array")

    # delay in samples (rounded to nearest integer)
    D = int(round(fs * (offset_ms / 1000.0)))
    if D < 0:
        raise ValueError("offset_ms must be >= 0")

    if not isinstance(num_delays, (int, np.integer)) or num_delays < 0:
        raise ValueError("num_delays must be an integer >= 0")

    # pick a safe dtype (avoid integer overflow when adding)
    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)  # promotes ints to float32+
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)

    pad = np.zeros(D, dtype=out_dtype)

    # start with the original padded once (like your original)
    orig = np.concatenate([x, pad])

    # add as many delays as requested
    new_s = orig.copy()
    for n in range(1, num_delays + 1):
        g = (gain / n) if scale_by_1_over_n else gain
        delay_n = np.concatenate([np.zeros(n * D, dtype=out_dtype), x])
        new_s = new_s + g * delay_n

    return new_s

## Filters
1. Create a function that will apply an feedforward comb filter by computing the delay length based off a given resonant frequency in Hz.

In [5]:
def feedforward_comb(x, fs, resonant_hz, gain=0.8, dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("x must be a 1-D array")
    if fs is None or fs <= 0:
        raise ValueError("fs must be a positive number (Hz)")
    if resonant_hz is None or resonant_hz <= 0:
        raise ValueError("resonant_hz must be a positive number (Hz)")
    if not np.isfinite(gain):
        raise ValueError("gain must be a finite number")

    D = int(round(fs / resonant_hz))
    if D < 1:
        raise ValueError("resonant_hz is too high for this fs (computed delay < 1 sample)")
    if D >= len(x):
        raise ValueError("delay is longer than or equal to the signal length")

    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)
    y = np.zeros_like(x, dtype=out_dtype)

    for n in range(len(x)):
        y[n] = x[n]
        if n - D >= 0:
            y[n] += gain * x[n - D]

    return y, D

2. Create a function that will apply an feedback comb filter by computing the delay length based off a given resonant frequency in Hz.

In [6]:
def feedback_comb(x, fs, resonant_hz, gain=0.8, dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("x must be a 1-D array")
    if fs is None or fs <= 0:
        raise ValueError("fs must be a positive number (Hz)")
    if resonant_hz is None or resonant_hz <= 0:
        raise ValueError("resonant_hz must be a positive number (Hz)")
    if not np.isfinite(gain):
        raise ValueError("gain must be a finite number")
    if abs(gain) >= 1:
        raise ValueError("For stability in a basic feedback comb, |gain| should be < 1")

    D = int(round(fs / resonant_hz))
    if D < 1:
        raise ValueError("resonant_hz is too high for this fs (computed delay < 1 sample)")
    if D >= len(x):
        raise ValueError("delay is longer than or equal to the signal length")

    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)
    y = np.zeros_like(x, dtype=out_dtype)

    for n in range(len(x)):
        y[n] = x[n]
        if n - D >= 0:
            y[n] += gain * y[n - D]

    return y, D

4. Use your comb filter functions on a wave from the audio folder. Try applying different resonant frequencies and delay lengths. How are the filter results different?

In [18]:

fs, x = read("../audio/dog_bark_dry.wav")

# If stereo, convert to mono
if x.ndim == 2:
    x = x.mean(axis=1)

# Try different resonant frequencies
freqs = 500
gain_ff = 0.7
gain_fb = 0.7


y_ff, Dff = feedforward_comb(x, fs, freqs, gain=gain_ff)
y_fb, Dfb = feedback_comb(x, fs, freqs, gain=gain_fb)

Audio(y_fb, rate=fs)


3. Create a function that will apply a butterworth filter to a signal with filter type options 'highpass', 'lowpass', 'bandpass', and 'bandstop'.

In [23]:
def butterworth_filter(x, fs, cutoff, order=4, btype='lowpass', dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        x = x.mean(axis=1)
    if fs is None or fs <= 0:
        raise ValueError("fs must be a positive number (Hz)")
    if order is None or order < 1 or int(order) != order:
        raise ValueError("order must be a positive integer")

    btype = btype.lower()
    valid = {'lowpass', 'highpass', 'bandpass', 'bandstop'}
    if btype not in valid:
        raise ValueError(f"btype must be one of {sorted(valid)}")

    nyq = fs / 2.0

    if btype in ('lowpass', 'highpass'):
        if cutoff is None or np.ndim(cutoff) != 0:
            raise ValueError("cutoff must be a single number (Hz) for lowpass/highpass")
        if not np.isfinite(cutoff) or cutoff <= 0 or cutoff >= nyq:
            raise ValueError(f"cutoff must be between 0 and Nyquist ({nyq} Hz)")
        Wn = cutoff / nyq
    else:
        if cutoff is None or len(cutoff) != 2:
            raise ValueError("cutoff must be a two-element sequence [low, high] (Hz) for bandpass/bandstop")
        low, high = float(cutoff[0]), float(cutoff[1])
        if not (np.isfinite(low) and np.isfinite(high)):
            raise ValueError("cutoff values must be finite numbers")
        if low <= 0 or high <= 0:
            raise ValueError("cutoff frequencies must be > 0")
        if low >= high:
            raise ValueError("For bandpass/bandstop, cutoff[0] must be < cutoff[1]")
        if high >= nyq:
            raise ValueError(f"Upper cutoff must be < Nyquist ({nyq} Hz)")
        Wn = [low / nyq, high / nyq]

    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)
    else:
        out_dtype = dtype
    x = x.astype(out_dtype, copy=False)

    b, a = butter(order, Wn, btype=btype, analog=False)

    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        raise ValueError(
            f"Signal too short for filtfilt with this order. "
            f"Need length > {padlen}, got {len(x)}. "
            f"Try lowering order."
        )

    y = filtfilt(b, a, x)
    return y

fs, x = read("../audio/AcousticGuitar.wav")
y_bs = butterworth_filter(x, fs, cutoff=[300, 3000], btype='bandpass')
Audio(y_bs, rate=fs)

## Reverb/Convolution

1. Create a function that will apply a simple moving average filter by convolving the filter kernel and an incoming signal.

In [27]:
def moving_average_filter(x, window_size, dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        x = x.mean(axis=1)
    if not isinstance(window_size, (int, np.integer)) or window_size < 1:
        raise ValueError("window_size must be a positive integer")

    if dtype is None:
        out_dtype = np.result_type(x.dtype, np.float32)
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)

    # moving average kernel
    h = np.ones(window_size, dtype=out_dtype) / window_size

    # convolution
    y = np.convolve(x, h, mode='same')

    return y

fs = 44100
dur = 0.5
x = np.random.normal(0, 1, int(fs*dur))
y = moving_average_filter(x, window_size=25)
Audio(y, rate=fs)

2. Apply your filter to a noise signal. What is the effect? What happens if you increase or decrease the kernel size?

In [ ]:
fs = 44100
dur = 0.5
x = np.random.normal(0, 1, int(fs*dur))
y = moving_average_filter(x, window_size=25)
Audio(y, rate=fs)

Applying a moving average filter to noise smooths it out by reducing the rapid, random fluctuations. When I increase the kernel size, the noise gets smoother and quieter, but more detail is lost and the signal feels more “blurred.” When I decrease the kernel size, less smoothing happens, so more of the original noise remains.

3. Create a function that applies convolution reverb to an input signal given an impulse response (this can be default loaded from the audio files). Use np.convolve to create this function.

In [29]:
def convolution_reverb(x, fs, ir=None, ir_path="../audio/impulse-response.wav",normalize_ir=True, normalize_out=True, dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        x = x.mean(axis=1)
    if fs is None or fs <= 0:
        raise ValueError("fs must be a positive number (Hz)")

    # Load impulse response from file if not provided
    if ir is None:
        ir_fs, ir = read(ir_path)
        if ir.ndim == 2:
            ir = ir.mean(axis=1)
        if ir_fs != fs:
            raise ValueError(f"Impulse response sample rate ({ir_fs}) must match input fs ({fs})")

    ir = np.asarray(ir)
    if ir.ndim != 1:
        raise ValueError("ir must be a 1-D array (mono)")
    if len(ir) < 1:
        raise ValueError("ir must not be empty")

    # Choose dtype (avoid integer overflow)
    if dtype is None:
        out_dtype = np.result_type(x.dtype, ir.dtype, np.float32)
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)
    ir = ir.astype(out_dtype, copy=False)

    # Optional: normalize IR so it doesn't blow up the level
    if normalize_ir:
        ir_max = np.max(np.abs(ir))
        if ir_max > 0 and np.isfinite(ir_max):
            ir = ir / ir_max

    # Convolution reverb
    y = np.convolve(x, ir, mode="full")

    # Optional: normalize output for playback
    if normalize_out:
        y_max = np.max(np.abs(y))
        if y_max > 0 and np.isfinite(y_max):
            y = y / y_max

    return y

fs, x = read("../audio/impulse-response.wav")
x = x.astype(np.float32)

y = convolution_reverb(x, fs, normalize_ir=True, normalize_out=True)

Audio(y, rate=fs)

4. Create another function that applies convolution reverb to an input signal given an impulse response, but this time do not use np.convolve. You should write the convolution from scratch.

    Challenge yourself to create the most efficient function and time your implementation against np.convolve. 
    While developing and testing your function, do not use real audio files. Start with short signals (e.g., impulses, noise, or short sinusoids). Using long signals with loop-based implementations will result in extremely slow run times.

    **Hint:** in a similar manner to how you may have designed your delay function, recall the functions `numpy.zeros` and `numpy.roll` along with how to manipulate multidimensional numpy arrays (e.g., scalar product, summing columns with `vstack`, etc.) If you plan to try `numpy.roll` DO NOT use it in a loop for convolution with a real audio file! (You'll kill your memory), instead consider the `map` function. You may also want to check out the following function which is similar to numpy.roll but more efficient for this task: `scipy.linalg.circulant`.

    You may wish to visit [here](https://numpy.org/doc/stable/user/basics.broadcasting.html) for review of broadcasting (i.e., form some calculation across index value I and column value C) in numpy

In [32]:
def convolution_reverb_manual(x, ir, dtype=None):
    x = np.asarray(x)
    ir = np.asarray(ir)

    if x.ndim != 1:
        x = x.mean(axis=1)
    if ir.ndim != 1:
        ir = ir.mean(axis=1)
    if len(ir) < 1:
        raise ValueError("ir must not be empty")

    if dtype is None:
        out_dtype = np.result_type(x.dtype, ir.dtype, np.float32)
    else:
        out_dtype = dtype

    x = x.astype(out_dtype, copy=False)
    ir = ir.astype(out_dtype, copy=False)

    N = len(x)
    M = len(ir)
    y = np.zeros(N + M - 1, dtype=out_dtype)

    # shift-and-add (loop over IR taps, vectorized over signal)
    for k in range(M):
        y[k:k+N] += ir[k] * x

    return y

# short impulse input
x = np.zeros(8, dtype=np.float32)
x[0] = 1.0

# short "impulse response"
ir = np.array([0.5, 0.25, 0.125], dtype=np.float32)

y_manual = convolution_reverb_manual(x, ir)
y_np = np.convolve(x, ir)

print("y_manual:", y_manual)
print("y_np:    ", y_np)

y_manual: [0.5   0.25  0.125 0.    0.    0.    0.    0.    0.    0.   ]
y_np:     [0.5   0.25  0.125 0.    0.    0.    0.    0.    0.    0.   ]
